# Gold — dim_fuente

Lee `source_system` y `source_file` de todas las tablas Silver y construye la dimensión de fuente.
Asigna `pipeline_name` y `nivel_agregacion` de forma manual por tabla de origen.

| Columna | Tipo | Descripción |
|---|---|---|
| `fuente_sk` | INT | Surrogate key |
| `source_system` | STRING | Sistema de origen (ej. `Dropbox/MSPAS`) |
| `source_file` | STRING | Archivo específico |
| `pipeline_name` | STRING | Notebook Silver que procesó esta fuente |
| `nivel_agregacion` | STRING | Granularidad geográfica: `municipal` \| `departamental` \| `nacional_edad` \| `nacional` |

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from functools import reduce
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

SILVER_SCHEMA = 'stage_silver_ss2'
GOLD_SCHEMA   = 'gold_ss2'
WRITE_MODE    = 'overwrite'

def get_job_run_id():
    try:
        return (
            dbutils.notebook.entry_point
            .getDbutils().notebook().getContext()
            .currentRunId().toString()
        )
    except Exception:
        return 'manual'

RUN_ID = get_job_run_id()
logger.info(f'Setup OK — run_id={RUN_ID}')

## Construcción de dim_fuente

UNION DISTINCT de `(source_system, source_file)` de las 10 tablas Silver.
El `pipeline_name` y `nivel_agregacion` se asignan según la tabla Silver de procedencia.

In [ ]:
# (silver_table, pipeline_name, nivel_agregacion)
_SILVER_TABLES = [
    ('ine_defunciones_sexo_edad',                              'stage_ine_edad',      'nacional_edad'),
    ('ine_defunciones_neonatales',                             'stage_ine_edad',      'nacional_edad'),
    ('ine_defunciones_postneonatales',                         'stage_ine_edad',      'nacional_edad'),
    ('ine_defunciones_depto_residencia',                       'stage_ine_geografia', 'departamental'),
    ('ine_defunciones_causas_externas',                        'stage_ine_geografia', 'departamental'),
    ('dbx_primeras_causas_de_morbilidad_2015_a_2025',         'stage_mspas',         'municipal'),
    ('dbx_morbilidad_enfermedades_cronicas_2015_a_2025',      'stage_mspas',         'municipal'),
    ('dbx_morbilidad_grupo_materno_infantil_2012_a_2025',     'stage_mspas',         'municipal'),
    ('who_guatemala',                                          'stage_who',           'nacional'),
    ('who_costa_rica',                                         'stage_who',           'nacional'),
]

dfs = []
for table, pipeline_name, nivel_agregacion in _SILVER_TABLES:
    df = (
        spark.table(f'{SILVER_SCHEMA}.{table}')
        .select('source_system', 'source_file')
        .distinct()
        .withColumn('pipeline_name',    F.lit(pipeline_name))
        .withColumn('nivel_agregacion', F.lit(nivel_agregacion))
    )
    dfs.append(df)
    logger.info(f'[{table}] fuentes distintas: {df.count()}')

df_union = reduce(lambda a, b: a.union(b), dfs).dropDuplicates()

df_dim = df_union.withColumn(
    'fuente_sk',
    F.row_number().over(Window.orderBy('source_system', 'source_file'))
)

df_dim = df_dim.select('fuente_sk', 'source_system', 'source_file', 'pipeline_name', 'nivel_agregacion')

logger.info(f'dim_fuente: {df_dim.count()} filas')

df_dim.write \
    .format('delta') \
    .mode(WRITE_MODE) \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{GOLD_SCHEMA}.dim_fuente')

logger.info(f'Escrito → {GOLD_SCHEMA}.dim_fuente')

## Validación

In [ ]:
df_val = spark.table(f'{GOLD_SCHEMA}.dim_fuente')
print(f'Total filas: {df_val.count()}')
df_val.show(truncate=False)

print('\n── Distribución por pipeline_name ──')
df_val.groupBy('pipeline_name', 'nivel_agregacion').count().orderBy('pipeline_name').show()